# 01 从零手写一个 Transformer decoder block

目标：用 PyTorch 小张量手写 decoder-only Transformer 的核心结构：embedding、causal self-attention、multi-head、residual、RMSNorm、SwiGLU MLP 和 lm head。

这不是高性能实现，只用于理解形状和数据流。


## 1. 安装依赖


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


## 2. 基础配置和输入


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

batch_size = 2
seq_len = 5
vocab_size = 32
hidden_size = 16
num_heads = 4
head_dim = hidden_size // num_heads
intermediate_size = 48

input_ids = torch.tensor([
    [1, 5, 7, 9, 2],
    [1, 4, 4, 8, 2],
])

print("input_ids:", tuple(input_ids.shape))
print("head_dim:", head_dim)


## 3. RMSNorm

很多 LLaMA/Qwen 类模型使用 RMSNorm。它不像 LayerNorm 那样减均值，而是按均方根缩放。


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


x = torch.randn(batch_size, seq_len, hidden_size)
rms_norm = RMSNorm(hidden_size)
layer_norm = nn.LayerNorm(hidden_size)

print("input mean:", x.mean(dim=-1)[0])
print("RMSNorm mean:", rms_norm(x).mean(dim=-1)[0])
print("LayerNorm mean:", layer_norm(x).mean(dim=-1)[0])
print("RMSNorm output shape:", tuple(rms_norm(x).shape))


## 4. Causal multi-head self-attention

关键形状：

```text
hidden: [batch, seq, hidden]
q/k/v : [batch, heads, seq, head_dim]
scores: [batch, heads, query_seq, key_seq]
```


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def split_heads(self, x):
        batch, seq, hidden = x.shape
        return x.view(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)

    def merge_heads(self, x):
        batch, heads, seq, head_dim = x.shape
        return x.transpose(1, 2).contiguous().view(batch, seq, heads * head_dim)

    def forward(self, hidden_states, return_weights=False):
        q = self.split_heads(self.q_proj(hidden_states))
        k = self.split_heads(self.k_proj(hidden_states))
        v = self.split_heads(self.v_proj(hidden_states))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(
            torch.ones(scores.shape[-2], scores.shape[-1], dtype=torch.bool, device=scores.device),
            diagonal=1,
        )
        scores = scores.masked_fill(causal_mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        output = self.o_proj(self.merge_heads(context))
        if return_weights:
            return output, weights
        return output


attention = CausalSelfAttention(hidden_size, num_heads)
attn_output, attn_weights = attention(x, return_weights=True)
print("attn_output:", tuple(attn_output.shape))
print("attn_weights:", tuple(attn_weights.shape))
print("head0 weights for row0:")
print(attn_weights[0, 0])
print("future attention mass:", float(torch.triu(attn_weights[0, 0], diagonal=1).sum()))


## 5. SwiGLU MLP

现代 decoder block 常见 MLP 是门控形式：`down_proj(silu(gate_proj(x)) * up_proj(x))`。


In [ ]:
class SwiGLUMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        gated = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(gated)


mlp = SwiGLUMLP(hidden_size, intermediate_size)
mlp_output = mlp(x)
print("mlp_output:", tuple(mlp_output.shape))


## 6. 拼成一个 decoder block

常见 pre-norm 结构：

```text
x = x + attention(norm(x))
x = x + mlp(norm(x))
```


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.input_norm = RMSNorm(hidden_size)
        self.self_attn = CausalSelfAttention(hidden_size, num_heads)
        self.post_attn_norm = RMSNorm(hidden_size)
        self.mlp = SwiGLUMLP(hidden_size, intermediate_size)

    def forward(self, x):
        x = x + self.self_attn(self.input_norm(x))
        x = x + self.mlp(self.post_attn_norm(x))
        return x


block = DecoderBlock(hidden_size, num_heads, intermediate_size)
block_output = block(x)
print("before block:", tuple(x.shape))
print("after block :", tuple(block_output.shape))
print("mean absolute delta:", float((block_output - x).abs().mean()))


## 7. 最小 decoder-only LM

embedding 把 token id 变成 hidden，decoder block 处理上下文，lm head 投影到 vocab logits。


In [ ]:
class TinyDecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_heads, intermediate_size, num_layers=2):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            DecoderBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids):
        hidden_states = self.embed_tokens(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        hidden_states = self.final_norm(hidden_states)
        logits = self.lm_head(hidden_states)
        return logits


tiny_lm = TinyDecoderOnlyLM(vocab_size, hidden_size, num_heads, intermediate_size)
logits = tiny_lm(input_ids)
print("logits:", tuple(logits.shape))
print("last-position logits shape:", tuple(logits[:, -1, :].shape))
print("next token ids:", logits[:, -1, :].argmax(dim=-1).tolist())


## 8. 检查 causal mask 的效果

如果 causal mask 正确，改变未来 token 不应该影响前面位置的 logits。


In [ ]:
changed = input_ids.clone()
changed[:, -1] = 17
with torch.inference_mode():
    original_logits = tiny_lm(input_ids)
    changed_logits = tiny_lm(changed)

# 前 4 个位置看不到最后一个 token，所以 logits 应该一致；最后一个位置会变。
past_diff = (original_logits[:, :-1, :] - changed_logits[:, :-1, :]).abs().max()
last_diff = (original_logits[:, -1, :] - changed_logits[:, -1, :]).abs().max()
print("max diff before changed future token:", float(past_diff))
print("max diff at changed position       :", float(last_diff))


## 面试总结

- decoder-only 模型每层通常是 pre-norm + causal self-attention + residual + MLP + residual。
- self-attention 的分数矩阵形状是 `[batch, heads, query_len, key_len]`。
- causal mask 保证第 t 个位置不能看未来 token。
- 多头注意力不是多个模型，而是把 hidden 拆成多个 head 并行看不同子空间。
- RMSNorm 只按均方根缩放，不做均值中心化；LayerNorm 会减均值再除标准差。
- SwiGLU 是门控 MLP，常见于 LLaMA/Qwen 等现代 decoder 架构。
- lm head 把 hidden state 投影到 vocab size，最后一个位置 logits 用来选下一个 token。
